In [1]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

chat = ChatOllama(model="llama3.2", temperature=0.7)

O poder da Composição com LCEL 

LCEL permite que você aninhe e combine cadeias de forma elegante
    . RunnablePassthrough: Passa a entrada para a próxima etapa sem modificação. Útil para reencaminhar dados.
    . RunnablePassthrough.assign(): Cria novas chaves no dicionário de entrada, permitindo que a saída de uma mini-cadeia seja usada em prompts futuros.
    . Funções Python como Runnable: Você pode transformar qualquer função python em um Runnable com RunnableLambda (ou simplesmente usando | diretamente, o LangChain converte a função automaticamente).

In [2]:
# Cadeia para extrair o sentimento
parser = StrOutputParser()
prompt_sentimento = ChatPromptTemplate.from_template("Analise o seguinte feedback e diga se o sentimento é 'positivo', 'negativo' ou 'neutro': {feedback}")
chain_sentimento = prompt_sentimento | chat | parser

In [3]:
# Cadeia para extrair o nome do produto
prompt_produto = ChatPromptTemplate.from_template("Qual o nome do produto mencionado no seguinte feedback? Responda apenas com o nome do produto: {feedback}")
chain_produto = prompt_produto | chat | parser

Usaremos o RunnablePassThrough para gerenciar as entradas e saídas e combinaremos tudo em uma unica Runnable.

In [4]:
def gerar_recomendacao(sentimento: str) -> str:
    """Função Python que gera uma recomendação de resposta com base no sentimento"""
    if sentimento.lower() == "positivo":
        return "Responda com gratidão e ofereça um cupom de desconto"
    elif sentimento.lower() == "negativo":
        return "Responda com empatia, peça desculpas e ofereça um suporte imediato."
    else:
        return "Agradeça o feedback e peça mais detalhes sobre a experiência"


# A função Python é agora parte da nossa cadeia
runnable_recomendacao = RunnablePassthrough.assign(
    recomendacao=lambda x: gerar_recomendacao(x['sentimento'])
)

A cadeia completa que orquestra todo o processo

1. Recebe o 'feedback'
2. Usa RunnablePassThrough.assign para chamar as duas sub-cadeias em paralelo
3. A saída é um dicionário com 'sentimento' e 'produto'.
4. A saída é passada para a próxima etapa, que é a nossa função python.
5. A função adiciona a chave 'recomendacao' ao dicionário

In [5]:
cadeia_completa = (
    RunnablePassthrough.assign(
        sentimento=chain_sentimento,
        produto=chain_produto
    )
    | runnable_recomendacao
)

In [ ]:
feedback_positivo = "Adorei o novo smartphone XPTO. A câmera é incrivel e o desempenho é excelente!"
print("--- Análise de Feedback Positivo ---")
# for chunk in cadeia_completa.stream({"feedback": feedback_positivo}):
#     print(chunk, end="", flush=True)
resultado_positivo = cadeia_completa.invoke({"feedback": feedback_positivo})

--- Análise de Feedback Positivo ---
{'feedback': 'Adorei o novo smartphone XPTO. A câmera é incrivel e o desempenho é excelente!'}

{'sentimento': 'O'}{'sentimento': ' sent'}{'sentimento': 'imento'}{'sentimento': ' do'}{'sentimento': ' feedback'}{'sentimento': ' é'}{'sentimento': " '"}{'sentimento': 'posit'}{'sentimento': 'ivo'}{'sentimento': "'."}{'sentimento': ''}{'sentimento': ''}{'produto': 'X'}{'produto': 'PT'}{'produto': 'O'}{'produto': ''}{'produto': ''}{'recomendacao': 'Agradeça o feedback e peça mais detalhes sobre a experiência'}